In [ ]:
import numpy as np
import opendssdirect as dss
import yadi.dss.model as dss_model
import yadi.dss.qsts as dss_qsts

# `yadi` Time series simulation examples: line and transformer currents

In [ ]:
#Load the case3_unbalanced file from PowerModelsDistribution.jl
# cktfile = '/home/sam/Research/yadi/test_cases/secondary_test_network/Master.DSS'
# cktfile = r"../test_cases/{case_file}".format(case_file=case_file)


##### 96 timestep case

cktfile = '/home/sam/Research/evacuationProject/data/OpenDSS Network/v1.1/2016/GSO/urban-suburban/scenarios/base_timeseries/opendss/uhs19_1247/Master_sample1.dss' 

##96 timestep case 

## Use `yadi` to run a time series simulation and construct datasets.

In [ ]:
qsts = dss_qsts.DSS_Timeseries(
    redirects=cktfile,
    time_step="15m",
    simulation_steps=24*4,
    simulation_mode="duty",
    simulation_controlmode='static',
    precompile=False
)


In [ ]:
qsts.compile_dss()
ratings = qsts.get_conductor_ratings()

In [ ]:
qsts.run()

In [ ]:
#Nodal current injections timeseries
I = qsts.currents_mvts

#Complex power injection timeseries
S = qsts.complex_powers_mvts

#Voltage phasor timeseries
V = qsts.voltages_mvts

#Voltage magnitude per unit (pu) timeseries
Vmags_pu = qsts.vmags_pu_mvts

#Line current timeseries
I_lines = qsts.line_currents_mvts

#Transformer current timeseries
#NOTE: Currently need to fix this, hold off on the transformer currents for now.
#I_xfmrs = qsts.xfmr_currents_mvts

In [ ]:
Vmags_pu[:,50]**2

In [ ]:
Vmags_pu.shape

In [ ]:
import matplotlib.pyplot as plt
plt.plot(Vmags_pu[:,100]**2)
plt.plot(Vmags_pu[:,50]**2)
plt.title("Controlmode=time, timestep=15m, simulation_steps=24*4")

## Compare different lengths of simulations for different dataset sizes

In [ ]:
qsts_24hr = dss_qsts.DSS_Timeseries(
    redirects=cktfile,
    time_step="15m",
    simulation_steps=24*4,
    simulation_mode="duty",
    simulation_controlmode='static',
    precompile=False
)
qsts_24hr.compile_dss()
qsts_24hr.run()
V_24hr = qsts_24hr.vmags_pu_mvts

qsts_48hr = dss_qsts.DSS_Timeseries(
    redirects=cktfile,
    time_step="15m",
    simulation_steps=48*4,
    simulation_mode="duty",
    simulation_controlmode='static',
    precompile=False
)
qsts_48hr.compile_dss()
qsts_48hr.run()
V_48hr = qsts_48hr.vmags_pu_mvts




In [ ]:
V_48hr[0:20,3]

In [ ]:
V_24hr[0:20,3]

In [ ]:
import matplotlib.pyplot as plt
plt.plot(V_48hr[:,3])
plt.plot(V_24hr[:,3])
plt.title("Allignment of time series for 48 and 24 hour simulations")

### Line Current Ratings

In [ ]:
ratings

In [ ]:
line_data = qsts.get_line_data()
line_data.keys()

In [ ]:
print('line_data length:',len(qsts.get_line_data()))
print('names length',len(dss.Lines.AllNames()))
print('I shape:',I_lines.shape)

In [ ]:
#Show line data
line_data = qsts.get_line_data()
line_data

In [ ]:
qsts.dss.CktElement.NumConductors()

In [ ]:
len(line_data)

### Note that the columns correspond to each single phase conductor, so the number of columns is much larger than the length of the line names.

In [ ]:
qsts.line_currents_mvts[0,0:100]

## Example 1: Get summary data about all lines and transformers in the network

In [ ]:
# Names
line_names = qsts.dss.Lines.AllNames()
xfmr_names = qsts.dss.Transformers.AllNames()

# Summary data
line_data = qsts.get_line_data()
xfmr_data = qsts.get_xfmr_data()

# Ratings 
line_ratings = qsts.get_line_ratings()
xfmr_ratings = qsts.get_xfmr_ratings()

# Show the summar data
print("Line data:")
print(line_data)
print("Transformer data:")
print(xfmr_data)

## Example 2: Show datasets of line currents and transformer currents (phasors)

In [ ]:
# Line currents -- note, these are the currents flowing w.r.t. from bus -> to bus
I_lines

## Example 2: Extract static data and current values for a single line or transformer at a single time step.

In [ ]:
xfmr_names = dss.Transformers.AllNames()
# Chooose a transformer name for the example
test_xfmr = xfmr_names[0]
print("All available transformer names:")
print(xfmr_names)
print("Transformer name chosen for example:")
print(test_xfmr)

In [ ]:
data = xfmr_data[test_xfmr]
print("Summary data for test xfmr name: {test_xfmr}".format(test_xfmr=test_xfmr))
data

In [ ]:
include_neutral = False # whether or not to include the neutral phase in the extracted currents
xfmr_currents = qsts.get_xfmr_currents(include_neutral=include_neutral) # NOTE: this is the internal function that retrieves currents at the current time step.
c = xfmr_currents[name]
print("Current matrix at transformer named: {x}:".format(x=name))
print(c)

In [ ]:

# Get from-> to and to->from currents.
c_from = c[0,:] # The first row is the from->to currents
c_to = c[1,:] # The second row is the to->from currents
print("From->to currents:")
print(c_from)
print("To->from currents:")
print(c_to)